# Qdrant Vector Database: Features, Architecture & Integration
This notebook introduces **Qdrant**, an open-source vector database built for high-performance similarity search and payload-based filtering. It covers Qdrant's internal architecture (Collections, Points, Named Vectors, and JSON Payloads) followed by an end-to-end hands-on pipeline with LangChain.

---

### Features of Qdrant

Qdrant is a **vector database** designed for efficient similarity search and Retrieval-Augmented Generation (RAG) applications.

It provides the following features:

- Collections
- Unique record IDs
- Dense vectors
- Sparse vectors
- Multiple named vectors
- JSON payloads
- Metadata indexes
- Insert, update, upsert, and delete operations
- Metadata filtering
- Persistence
- Snapshots
- Replication and sharding
- HTTP and gRPC server APIs

---

### Qdrant Architecture

```text
Qdrant Database
│
├── Collection: company-documents
│     │
│     ├── Point 1
│     │     ├── ID
│     │     ├── Dense Vector
│     │     └── Payload (Metadata)
│     │
│     ├── Point 2
│     │     ├── ID
│     │     ├── Dense Vector
│     │     └── Payload (Metadata)
│     │
│     └── Point 3
│
├── Vector Index
│     └── HNSW (or other supported retrieval indexes)
│
├── Payload Index
│     └── category, source, page, user_id, ...
│
└── Storage
      ├── Memory
      └── Disk
```

---

### Simple Classroom Explanation

A **Qdrant collection** is similar to a table in a traditional database.

Each record in the collection is called a **point**, and every point contains:

- A unique **ID**
- One or more **vectors** (dense, sparse, or named vectors)
- A **payload** containing metadata (stored as JSON)

Qdrant automatically builds vector indexes (such as **HNSW**) for fast similarity search and payload indexes for efficient metadata filtering. It also supports persistence, snapshots, replication, sharding, and exposes HTTP and gRPC APIs for production deployments.

## 1. Setup & Library Imports
Import required classes from `qdrant_client`, `langchain_qdrant`, `langchain_community`, `langchain_google_genai`, and standard Python utilities.

In [1]:
# Load path and document loading utilities
from __future__ import annotations
from pathlib import Path
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader

C:\Users\ahmad\AppData\Local\Temp\ipykernel_10288\583762284.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:307: UserWarning: Failed to initialize NumPy: module 'numpy._globals' has no attribute '_signature_descriptor' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method

In [2]:
# Import Qdrant client, models, and text splitter
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

## 2. Document Ingestion, Chunking & Payload Metadata
Load the PDF document using `PyPDFLoader`, split it into manageable chunks with `RecursiveCharacterTextSplitter`, and assign custom payload metadata (`chunk_id`, `filename`) to each chunk.

In [3]:
# Load PDF document using PyPDFLoader
loader = PyPDFLoader(r"D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\03_Vector_Databases\data\llama2-research-paper.pdf")
pages = loader.load()

print("PDF pages loaded:", len(pages))

PDF pages loaded: 77


In [4]:
# Split loaded pages into overlapping text chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=120,
)
chunks = text_splitter.split_documents(pages)
print("Chunks created:", len(chunks))

Chunks created: 174


In [5]:
# Enrich chunk metadata dictionaries with payload attributes
for chunk_index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_index
    chunk.metadata["filename"] = "llama2-research-paper.pdf"

## 3. Initialize Embedding Model & Check Dimension
Instantiate `GoogleGenerativeAIEmbeddings` and verify vector output dimensionality.

In [6]:
# Initialize Google Generative AI embeddings and inspect vector dimension
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
dimension = len(
    embedding_model.embed_query("dimension check")
)
print("Embedding dimension:", dimension)

Embedding dimension: 3072


## 4. Qdrant Client Initialization (Cloud & Local Options)
Authenticate with Qdrant Cloud using API key and cluster endpoint (or optionally initialize a local file-backed Qdrant instance).

In [7]:
# Optional: Local Qdrant instance initialization (commented out)
# client = QdrantClient(
#     path=str(Path(__file__).parent / "qdrant_data")
# )

In [8]:
# Load environment variables for Qdrant Cloud authentication
import os
from dotenv import load_dotenv
load_dotenv()
qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_cluster_endpoint = os.getenv("QDRANT_Cluster_Endpoint")

In [9]:
# Instantiate QdrantClient connected to Qdrant Cloud cluster
client = QdrantClient(api_key=qdrant_api_key, url=qdrant_cluster_endpoint)

## 5. Collection Creation & Vector Configuration
Define collection name (`company-policy-rag`) and provision the Qdrant collection with Cosine distance metric if it does not exist.

In [10]:
# Define target Qdrant collection name
COLLECTION_NAME = "company-policy-rag"

In [11]:
# Create Qdrant collection configured with Cosine distance metric and vector size
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=dimension,
            distance=models.Distance.COSINE,
        ),
    )

## 6. Wrap Qdrant in LangChain VectorStore & Generate Point IDs
Instantiate `QdrantVectorStore` and generate stable UUIDs for each point record.

In [12]:
# Wrap Qdrant client in LangChain QdrantVectorStore
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embedding_model,
)

In [13]:
# Generate unique UUIDs for each point record
chunk_ids = [
    str(uuid4())
    for _ in chunks
]

## 7. Batch Point Upsertion with Rate Limiting
Upsert document vectors, point IDs, and payloads into Qdrant in rate-limited batches with retry logic for 429 errors.

In [15]:
# Add documents in batches with automatic retries on rate limit errors
import time

batch_size = 20
total_inserted = 0

for i in range(0, len(chunks), batch_size):
    print(f"Adding batch {i} to {i+batch_size} of {len(chunks)}...")
    batch_chunks = chunks[i:i+batch_size]
    batch_ids = chunk_ids[i:i+batch_size]
    
    success = False
    while not success:
        try:
            inserted = vector_store.add_documents(
                documents=batch_chunks,
                ids=batch_ids,
            )
            total_inserted += len(inserted)
            success = True
            if i + batch_size < len(chunks):
                time.sleep(15)
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                print("Rate limit hit! Pausing for 60 seconds before retrying...")
                time.sleep(60)
            else:
                raise e

print("Inserted chunks:", total_inserted)

Adding batch 0 to 20 of 174...
Adding batch 20 to 40 of 174...
Adding batch 40 to 60 of 174...
Adding batch 60 to 80 of 174...
Adding batch 80 to 100 of 174...
Adding batch 100 to 120 of 174...
Adding batch 120 to 140 of 174...
Adding batch 140 to 160 of 174...
Adding batch 160 to 180 of 174...
Inserted chunks: 174


## 8. Semantic Similarity Search & Payload Inspection
Execute similarity search queries against Qdrant and inspect retrieved point contents along with JSON metadata payloads.

In [16]:
# Perform similarity search and display results with metadata payload
query = "What is the annual leave policy?"

results = vector_store.similarity_search(
    query=query,
    k=4,
)

for rank, document in enumerate(results, start=1):
    print(f"\nResult {rank}")
    print("Content:", document.page_content)
    print("Metadata:", document.metadata)


Result 1
Content: presented to the annotators. From this data, we report wins, ties, and losses in our results. Three annotators
rateeachgenerationpair. Priorexperimentswithfiveannotatorsdidnotchangetheresultsorinter-annotator
agreement significantly.
57
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\Coding\\Full-Stack-GenAI-AgenticAI-Bootcamp\\05_RAG\\03_Vector_Databases\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 56, 'page_label': '57', 'chunk_id': 127, 'filename': 'llama2-research-paper.pdf', '_id': '98371500-ff5e-4122-b79c-31386da8d60d', '_collection_name': 'company-policy-rag'}

Result 2
Content: on pizza, I think they are an abomination and should nev

## 9. Retriever Conversion & Context Assembly
Convert `QdrantVectorStore` into a LangChain retriever and format retrieved context strings for RAG pipelines.

In [17]:
# Convert vector store to retriever and build RAG context string
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

context_documents = retriever.invoke(
    "What benefits are available to employees?"
)

context = "\n\n".join(
    document.page_content
    for document in context_documents
)

print(context)

on pizza, I think they are an abomination and should never be allowed on any
self-respecting pizza. So, if you’re looking for someone to share a slice with,
I’m your girl! Just make sure you meet my high standards for pizza excellence.
0.62 0.83
25% Sure,I’dloveto! Butlet’sgotoaplacethatservesrealChicago-stylepizza,with
a thick crust and a hearty topping of cheese and tomato sauce. And please,
no pineapple! That’s just not what pizza is all about. And by the way, I think
it’s important to respect the integrity of the pizza. It should never be folded, it
should be eaten in slices. Folding it just makes it soggy and messy. So, let’s go
find a good pizza joint that serves the real deal. I’m starving!
0.74 0.91
50% I’m glad you asked! I would love to go grab a slice of pizza after work today.
But let me tell you, I have some strong opinions about pizza. First of all, I think
Chicago pizza is the best. It’s deep dish, it’s thick, it’s cheesy, and it’s just all
around delicious. I can’t stan